# Виклик: Аналіз тексту про науку про дані

У цьому прикладі давайте зробимо просту вправу, яка охоплює всі кроки традиційного процесу науки про дані. Вам не потрібно писати жодного коду, ви можете просто клацнути по клітинках нижче, щоб виконати їх і спостерігати результат. Як виклик, вам пропонується спробувати цей код з різними даними.

## Мета

У цьому уроці ми обговорювали різні концепції, пов’язані з наукою про дані. Давайте спробуємо відкрити більше пов'язаних концепцій, зробивши **текстовий майнінг**. Ми почнемо з тексту про науку про дані, витягнемо з нього ключові слова, а потім спробуємо візуалізувати результат.

Як текст я використаю сторінку про науку про дані з Вікіпедії:


In [ ]:
url = 'https://en.wikipedia.org/wiki/Data_science'

## Крок 1: Отримання даних

Перший крок у будь-якому процесі наукових досліджень даних – отримання даних. Для цього ми використовуватимемо бібліотеку `requests`:


In [ ]:
import requests

# Define a custom header.
headers = {
    'User-Agent': 'DataScienceChallenge/1.0 (myemail@gmail.com)'
}

# Pass the headers into the get request
response = requests.get(url, headers=headers)

if response.status_code == 200:
    text = response.content.decode('utf-8')
    print(text[:1000])
else:
    print(f"Error: {response.status_code}")

## Крок 2: Перетворення даних

Наступним кроком є перетворення даних у форму, придатну для обробки. У нашому випадку ми завантажили HTML-код сторінки, і нам потрібно перетворити його у звичайний текст.

Існує багато способів зробити це. Ми будемо використовувати [BeautifulSoup](https://www.crummy.com/software/BeautifulSoup/), популярну бібліотеку Python для парсингу HTML. BeautifulSoup дозволяє нам орієнтуватися на конкретні HTML-елементи, щоб зосередитися на основному вмісті статті з Вікіпедії та зменшити кількість меню навігації, бічних панелей, підвалів та іншого неістотного вмісту (хоча деякий стандартний текст може залишатися).


Спочатку нам потрібно встановити бібліотеку BeautifulSoup для парсингу HTML:


In [ ]:
import sys
!{sys.executable} -m pip install beautifulsoup4

In [ ]:
from bs4 import BeautifulSoup

# Parse the HTML content
soup = BeautifulSoup(text, 'html.parser')

# Extract only the main article content from Wikipedia
# Wikipedia uses 'mw-parser-output' class for the main article content
content = soup.find('div', class_='mw-parser-output')

def clean_wikipedia_content(content_node):
    """Remove common non-article elements from a Wikipedia content node."""
    # Strip jump links, navboxes, reference lists/superscripts, edit sections, TOC, sidebars, etc.
    selectors = [
        '.mw-jump-link',
        '.navbox',
        '.reflist',
        'sup.reference',
        '.mw-editsection',
        '.hatnote',
        '.metadata',
        '.infobox',
        '#toc',
        '.toc',
        '.sidebar',
    ]
    for selector in selectors:
        for el in content_node.select(selector):
            el.decompose()

if content:
    # Clean the content node to better approximate article text only.
    clean_wikipedia_content(content)
    text = content.get_text(separator=' ', strip=True)
    print(text[:1000])
else:
    print("Could not find main content. Using full page text.")
    text = soup.get_text(separator=' ', strip=True)
    print(text[:1000])

## Крок 3: Отримання Інсайтів

Найважливішим кроком є перетворення наших даних у форму, з якої ми можемо отримати інсайти. У нашому випадку ми хочемо витягнути ключові слова з тексту та побачити, які ключові слова є більш значущими.

Ми використаємо бібліотеку Python під назвою [RAKE](https://github.com/aneesha/RAKE) для витягання ключових слів. Спочатку давайте встановимо цю бібліотеку, якщо вона ще не встановлена: 


In [ ]:
import sys
!{sys.executable} -m pip install nlp_rake

Основна функціональність доступна через об’єкт `Rake`, який ми можемо налаштувати, використовуючи деякі параметри. У нашому випадку ми встановимо мінімальну довжину ключового слова в 5 символів, мінімальну частоту ключового слова в документі - 3, а максимальну кількість слів у ключовому слові - 2. Не соромтеся експериментувати з іншими значеннями та спостерігати результат.


In [ ]:
import nlp_rake
extractor = nlp_rake.Rake(max_words=2,min_freq=3,min_chars=5)
res = extractor.apply(text)
res


Ми отримали список термінів разом із відповідним ступенем важливості. Як ви бачите, найбільш релевантні дисципліни, такі як машинне навчання та великі дані, знаходяться в списку на верхніх позиціях.

## Крок 4: Візуалізація Результату

Люди краще сприймають дані у візуальній формі. Тому часто має сенс візуалізувати дані, щоб зробити певні висновки. Ми можемо використати бібліотеку `matplotlib` у Python для побудови простого розподілу ключових слів із їх релевантністю:


In [ ]:
import matplotlib.pyplot as plt

def plot(pair_list):
    k,v = zip(*pair_list)
    plt.bar(range(len(k)),v)
    plt.xticks(range(len(k)),k,rotation='vertical')
    plt.show()

plot(res)

Однак існує ще кращий спосіб візуалізувати частоти слів — використання **Хмари слів**. Нам потрібно буде встановити іншу бібліотеку, щоб побудувати хмару слів із нашого списку ключових слів.


In [ ]:
!{sys.executable} -m pip install wordcloud

Об'єкт `WordCloud` відповідає за прийом або оригінального тексту, або попередньо обчисленого списку слів з їх частотами, і повертає зображення, яке потім можна відобразити за допомогою `matplotlib`:


In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt

wc = WordCloud(background_color='white',width=800,height=600)
plt.figure(figsize=(15,7))
plt.imshow(wc.generate_from_frequencies({ k:v for k,v in res }))

Ми також можемо передати оригінальний текст до `WordCloud` — подивимося, чи зможемо отримати подібний результат:


In [ ]:
plt.figure(figsize=(15,7))
plt.imshow(wc.generate(text))

In [ ]:
wc.generate(text).to_file('images/ds_wordcloud.png')

Ви можете побачити, що хмара слів тепер виглядає більш вражаюче, але вона також містить багато шуму (наприклад, нерелевантні слова, такі як `Retrieved on`). Крім того, ми отримуємо менше ключових слів, що складаються з двох слів, таких як *data scientist* або *computer science*. Це тому, що алгоритм RAKE значно краще справляється з вибором хороших ключових слів із тексту. Цей приклад ілюструє важливість попередньої обробки та очищення даних, оскільки чітка картина наприкінці дозволить нам приймати кращі рішення.

У цьому завданні ми пройшли простий процес вилучення сенсу з тексту Вікіпедії у вигляді ключових слів і хмари слів. Цей приклад досить простий, але він добре демонструє всі типові кроки, які виконуватиме дата-сайєнтист при роботі з даними — починаючи від отримання даних і закінчуючи візуалізацією.

У нашому курсі ми обговоримо всі ці кроки детально.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Відмова від відповідальності**:
Цей документ було перекладено за допомогою сервісу штучного інтелекту для перекладу [Co-op Translator](https://github.com/Azure/co-op-translator). Хоча ми прагнемо до точності, будь ласка, майте на увазі, що автоматичні переклади можуть містити помилки або неточності. Оригінальний документ рідною мовою слід вважати авторитетним джерелом. Для критично важливої інформації рекомендується професійний людський переклад. Ми не несемо відповідальності за будь-які непорозуміння або неправильні тлумачення, що виникли внаслідок використання цього перекладу.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
